In [14]:
import numpy as np
from tqdm import tqdm
#load in txt
data = np.loadtxt("/Users/janicesubroto/Desktop/Lu Lab/Python Quiz/Q2_sample_barcode_data.txt", dtype=str) #array of sequences

In [ ]:
#look at avg length of each sequence
"""
lengths = []
for d in data:
    #print(len(d))
    lengths.append(len(d))

#check if they're all 50 bp in length
def same(a):
    return all(x == a[0] for x in a)

print(same(lengths))
"""
#all sequences are 50bp in length


True


In [16]:
#levenshtein distance
def edit_distance(s1, s2):
    m = len(s1)
    n = len(s2)

    mat = [[0 for _ in range(n + 1)] for _ in range(m + 1)] #m+1 rows, n+1 columns
    for i in range(m+1):
        mat[i][0] = i

    for j in range(n+1):
        mat[0][j] = j

    #fill matrix
    for i in range(1, m+1):
        for j in range(1, n+1):
            if s1[i-1] == s2[j-1]: #no difference
                mat[i][j] = mat[i-1][j-1]

            else:
                mat[i][j] = 1 + min(
                    mat[i][j-1],
                    mat[i-1][j],
                    mat[i-1][j-1]
                )

    return mat[m][n]

In [ ]:
#generate a dictionary of signatures, where the IDs are kmers, and the value are candidate clusters.
#This would minimize the space that each sequence needs to be compared to
from inspect import signature


signatures = {} #kmer: cluster sequences
clusters = {} #cluster: counts
k = 10 #kmer size

#for each sequence
#first break up into kmers
#check if kmers exist in signatures
#if yes, then compare the sequence with the cluster using edit_distance
#if no, then add this new sequence to cluster and initialize it with count 1
for sequence in tqdm(data):

    seq_kmers = [sequence[i:i+k] for i in range(len(sequence) - k + 1)]
    match = False

    #check if kmers exist in signatures
    for kmer in seq_kmers:
        if kmer in signatures:
            #check the sequence against the clusters in signatures
            candidate_clusters = signatures[kmer] #extract clusters

            for c in candidate_clusters:
                if edit_distance(c, sequence) <= 2:
                    if c in clusters:
                        clusters[c] += 1
                        match = True
                        break

            if match == True:
                break

    if not match: #this means none of the kmers worked, and they need to be added to signatures
        clusters[sequence] = 1 #if there's no match then you also need to initialize sequence as a new cluster
        for kmer in seq_kmers:
            if kmer in signatures:
                signatures[kmer].append(sequence)
            else:
                signatures[kmer] = []
                signatures[kmer].append(sequence)


In [ ]:
clusters